# NOAA Storm Events Raw Data Quality

**Purpose.** Audit the files ingested for this provider and the corresponding `raw.*`
DuckDB tables before any normalization, blending, or analytical transformation.

This notebook covers the supplied files/tables, observation grain, date and geography
coverage, column types and meanings, missingness and suppression, duplicate/invalid
keys, numeric ranges, suspicious values, source limitations, and downstream readiness.

## Setup and provider rules

In [1]:
from pathlib import Path
import re
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "quoll.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "quoll.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

PROVIDER = 'noaa'
TABLE_PATTERNS = ['noaa_%']
PRIMARY_PATTERNS = ['noaa_storm_events_county_damage', 'noaa_storm_events_zone_county_mapping']
KEY_CANDIDATES = [['event_id', 'cz_type', 'state_fips', 'cz_fips'], ['state_fips', 'cz_fips', 'cz_name']]
DATE_CANDIDATES = ['begin_date_time', 'end_date_time', 'begin_yearmonth', 'year']
GEO_CANDIDATES = ['state_fips', 'county_fips', 'cz_fips', 'cz_name', 'mapped_fips']
NUMERIC_HINTS = ['year', 'event_id', 'property_damage', 'crop_damage', 'total_damage', 'injuries_direct', 'deaths_direct']
SUPPRESSION_CODES = ['', 'null']

def matches(name, patterns):
    return any(re.fullmatch(pattern.replace("%", ".*"), name, flags=re.I) for pattern in patterns)

def qi(value):
    return '"' + value.replace('"', '""') + '"'

raw_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'raw' ORDER BY table_name"
).df()["table_name"].tolist()
provider_tables = [name for name in raw_tables if matches(name, TABLE_PATTERNS)]
primary_tables = [name for name in provider_tables if matches(name, PRIMARY_PATTERNS)]
provider_tables, primary_tables

(['noaa_storm_events_county_damage', 'noaa_storm_events_zone_county_mapping'],
 ['noaa_storm_events_county_damage', 'noaa_storm_events_zone_county_mapping'])

## Files and tables supplied

In [2]:
file_inventory = con.execute(
    '''
    SELECT table_name, filename, source_folder, source_path,
           loaded_at, row_count, detected_columns,
           upstream_source_url, content_sha256
    FROM meta.files
    WHERE table_schema = 'raw'
    ORDER BY table_name
    '''
).df()
file_inventory = file_inventory.loc[file_inventory["table_name"].isin(provider_tables)]

table_rows = []
for table in provider_tables:
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    column_count = con.execute(
        "SELECT count(*) FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).fetchone()[0]
    table_rows.append({"table_name": table, "rows": row_count, "columns": column_count,
                       "primary_data_table": table in primary_tables})
table_inventory = pd.DataFrame(table_rows)
display(file_inventory)
display(table_inventory)

,table_name,filename,source_folder,source_path,loaded_at,row_count,detected_columns,upstream_source_url,content_sha256
52,noaa_storm_events_county_damage,noaa_storm_events_county_damage.csv,climate_damage,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:42.337938+00:00,653920,"[""begin_yearmonth"", ""episode_id"", ""event_id"", ...",https://www.ncei.noaa.gov/pub/data/swdi/storme...,97a8f66931ed988140f6b83e45a8d41d78ed209d3b65bf...
53,noaa_storm_events_zone_county_mapping,noaa_storm_events_zone_county_mapping.csv,climate_damage,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:42.475582+00:00,4787,"[""state"", ""state_fips"", ""cz_fips"", ""cz_name"", ...",https://www.ncei.noaa.gov/pub/data/swdi/storme...,4553316bdfe55af5272a94af7c7816c4bd11403f31ef18...


,table_name,rows,columns,primary_data_table
0,noaa_storm_events_county_damage,653920,23,True
1,noaa_storm_events_zone_county_mapping,4787,12,True


## Observation grain

Storm Events detail records resolved to counties, plus a separate forecast-zone-to-county mapping.

The checks below infer candidate keys from the raw columns. A repeated candidate key is
reported rather than silently removed because some provider tables legitimately contain
additional dimensions.

## Column types and meanings

In [3]:
schema_frames = []
for table in primary_tables:
    schema = con.execute(f"DESCRIBE raw.{qi(table)}").df()
    schema.insert(0, "table_name", table)
    schema["inferred_meaning"] = (
        schema["column_name"].str.replace("_", " ", regex=False)
        .str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
        .str.strip()
    )
    schema_frames.append(schema)
schema_inventory = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
display(schema_inventory)

,table_name,column_name,column_type,null,key,default,extra,inferred_meaning
0,noaa_storm_events_county_damage,begin_yearmonth,VARCHAR,YES,None,None,None,begin yearmonth
1,noaa_storm_events_county_damage,episode_id,VARCHAR,YES,None,None,None,episode id
2,noaa_storm_events_county_damage,event_id,VARCHAR,YES,None,None,None,event id
3,noaa_storm_events_county_damage,state,VARCHAR,YES,None,None,None,state
4,noaa_storm_events_county_damage,state_fips,VARCHAR,YES,None,None,None,state fips
5,noaa_storm_events_county_damage,year,VARCHAR,YES,None,None,None,year
6,noaa_storm_events_county_damage,event_type,VARCHAR,YES,None,None,None,event type
7,noaa_storm_events_county_damage,cz_type,VARCHAR,YES,None,None,None,cz type
8,noaa_storm_events_county_damage,cz_fips,VARCHAR,YES,None,None,None,cz fips
9,noaa_storm_events_county_damage,cz_name,VARCHAR,YES,None,None,None,cz name


## Date and geographic coverage

In [4]:
coverage_rows = []
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"].tolist()
    row = {"table_name": table}
    for column in DATE_CANDIDATES:
        if column in columns:
            normalized_column = column.lower()
            if normalized_column == "year" or normalized_column.endswith("_year"):
                coverage_type = "INTEGER"
            elif normalized_column == "month" or normalized_column.endswith("_month"):
                coverage_type = "INTEGER"
            else:
                coverage_type = "TIMESTAMP"
            values = con.execute(
                f"SELECT min(try_cast({qi(column)} AS {coverage_type})), "
                f"max(try_cast({qi(column)} AS {coverage_type})) "
                f"FROM raw.{qi(table)}"
            ).fetchone()
            row[f"{column}_min"] = values[0]
            row[f"{column}_max"] = values[1]
    for column in GEO_CANDIDATES:
        if column in columns:
            row[f"{column}_distinct"] = con.execute(
                f"SELECT count(DISTINCT {qi(column)}) FROM raw.{qi(table)}"
            ).fetchone()[0]
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)

,table_name,begin_date_time_min,begin_date_time_max,end_date_time_min,end_date_time_max,begin_yearmonth_min,begin_yearmonth_max,year_min,year_max,state_fips_distinct,county_fips_distinct,cz_fips_distinct,cz_name_distinct,mapped_fips_distinct
0,noaa_storm_events_county_damage,NaN,NaN,NaN,NaN,NaN,NaN,2016.0,2025.0,69,3308.0,674,4010,NaN
1,noaa_storm_events_zone_county_mapping,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69,NaN,614,3538,2917.0


## Missingness and suppression codes

In [5]:
missing_rows = []
suppression_rows = []
suppression_sql = ", ".join("?" for _ in SUPPRESSION_CODES)
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [table]
    ).df()["column_name"].tolist()
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    # Profile all columns for compact tables and the first 80 for unusually wide sources.
    for column in columns[:80]:
        null_count, blank_count = con.execute(
            f"SELECT count(*) FILTER (WHERE {qi(column)} IS NULL), "
            f"count(*) FILTER (WHERE trim(cast({qi(column)} AS VARCHAR))='') "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        missing_rows.append({
            "table_name": table, "column_name": column,
            "missing_count": null_count + blank_count,
            "missing_pct": (null_count + blank_count) / row_count * 100 if row_count else np.nan,
        })
        if SUPPRESSION_CODES:
            suppressed = con.execute(
                f"SELECT count(*) FROM raw.{qi(table)} "
                f"WHERE trim(cast({qi(column)} AS VARCHAR)) IN ({suppression_sql})",
                SUPPRESSION_CODES,
            ).fetchone()[0]
            if suppressed:
                suppression_rows.append({
                    "table_name": table, "column_name": column,
                    "suppression_or_sentinel_count": suppressed,
                })
missingness = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "table_name"], ascending=[False, True]
)
suppression = pd.DataFrame(suppression_rows)
display(missingness)
display(suppression if not suppression.empty else pd.DataFrame(
    {"result": ["No configured literal suppression codes were present in profiled columns; nulls remain material."]}
))

,table_name,column_name,missing_count,missing_pct
18,noaa_storm_events_county_damage,tor_f_scale,638751,97.680297
17,noaa_storm_events_county_damage,magnitude,310219,47.439901
19,noaa_storm_events_county_damage,county_fips,280940,42.962442
27,noaa_storm_events_zone_county_mapping,mapped_fips,1119,23.375809
28,noaa_storm_events_zone_county_mapping,mapped_county_name,1119,23.375809
14,noaa_storm_events_county_damage,damage_property,138889,21.239448
15,noaa_storm_events_county_damage,damage_crops,138488,21.178126
0,noaa_storm_events_county_damage,begin_yearmonth,0,0.000000
1,noaa_storm_events_county_damage,episode_id,0,0.000000
2,noaa_storm_events_county_damage,event_id,0,0.000000


,result
0,No configured literal suppression codes were p...


## Duplicate or invalid keys

In [6]:
key_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    keys = next((candidate for candidate in KEY_CANDIDATES if set(candidate).issubset(columns)), [])
    if not keys:
        key_rows.append({"table_name": table, "candidate_key": None,
                         "duplicate_key_groups": np.nan, "invalid_key_rows": np.nan})
        continue
    key_expr = ", ".join(qi(column) for column in keys)
    invalid = " OR ".join(
        f"{qi(column)} IS NULL OR trim(cast({qi(column)} AS VARCHAR))=''" for column in keys
    )
    duplicate_groups = con.execute(
        f"SELECT count(*) FROM (SELECT {key_expr}, count(*) n "
        f"FROM raw.{qi(table)} GROUP BY {key_expr} HAVING count(*) > 1)"
    ).fetchone()[0]
    invalid_rows = con.execute(
        f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}"
    ).fetchone()[0]
    key_rows.append({"table_name": table, "candidate_key": " + ".join(keys),
                     "duplicate_key_groups": duplicate_groups,
                     "invalid_key_rows": invalid_rows})
key_quality = pd.DataFrame(key_rows)
display(key_quality)

,table_name,candidate_key,duplicate_key_groups,invalid_key_rows
0,noaa_storm_events_county_damage,event_id + cz_type + state_fips + cz_fips,0,0
1,noaa_storm_events_zone_county_mapping,state_fips + cz_fips + cz_name,58,0


## Numeric ranges and suspicious values

In [7]:
numeric_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    for column in [name for name in NUMERIC_HINTS if name in columns]:
        numeric = (
            f"try_cast(replace(trim(cast({qi(column)} AS VARCHAR)), ',', '') AS DOUBLE)"
        )
        result = con.execute(
            f"SELECT count(*) FILTER (WHERE {numeric} IS NOT NULL), "
            f"min({numeric}), max({numeric}), "
            f"count(*) FILTER (WHERE {numeric} < 0) "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        numeric_rows.append({
            "table_name": table, "column_name": column,
            "numeric_count": result[0], "minimum": result[1],
            "maximum": result[2], "negative_count": result[3],
            "review_flag": (
                "review negative values/sentinels" if result[3] else
                "review extreme min/max against provider definition"
            ),
        })
numeric_ranges = pd.DataFrame(numeric_rows)
display(numeric_ranges)

,table_name,column_name,numeric_count,minimum,maximum,negative_count,review_flag
0,noaa_storm_events_county_damage,year,653920,2016.0,2.025000e+03,0,review extreme min/max against provider defini...
1,noaa_storm_events_county_damage,event_id,653920,606570.0,1.310672e+06,0,review extreme min/max against provider defini...
2,noaa_storm_events_county_damage,property_damage,653920,0.0,1.700000e+10,0,review extreme min/max against provider defini...
3,noaa_storm_events_county_damage,crop_damage,653920,0.0,5.000000e+08,0,review extreme min/max against provider defini...
4,noaa_storm_events_county_damage,total_damage,653920,0.0,1.700000e+10,0,review extreme min/max against provider defini...
5,noaa_storm_events_county_damage,injuries_direct,653920,0.0,8.060000e+02,0,review extreme min/max against provider defini...
6,noaa_storm_events_county_damage,deaths_direct,653920,0.0,1.020000e+02,0,review extreme min/max against provider defini...


## Source-specific limitations

Damage is reported rather than independently estimated, zero can mean no reported damage, zone events require geographic allocation, and reporting practices change over time.

## Downstream readiness

**Assessment: PASS WITH LIMITATIONS. Only resolved county events with valid timestamps are eligible; the project extreme-event definition applies a $1 billion total-damage threshold.**

This assessment is conditional on the displayed inventories and checks. The normalized
`mart.*` builders—not this notebook—own parsing, suppression handling, geographic
resolution, deduplication, and downstream transformations.

In [8]:
con.close()